<a href="https://colab.research.google.com/github/EsarFatima/MachineLearning-flyrank-/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/EsarFatima/MachineLearning-flyrank-"
REPO_DIR = "MachineLearning-flyrank-"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down.
# trend_direction / trend_pct are ONLY used here to build the label for review/evaluation.
# They are never used as scoring inputs below (that would be leakage).
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df.shape[0], "pages |  base decline rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  base decline rate: 0.542


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Plain-word rule:** A page is worth reviewing if it hasn't been touched in a while (**stale**)
and it's still pulling a meaningful, but not freakish, amount of search traffic (**visible**).
Old and dead is not worth the time. Fresh and slipping is not this rule's job. Stale-and-still-seen
is the sweet spot — someone already invested in it, and it's still in front of people.

**Reason code (one, always the same):** `stale_visible_decline_risk`

**Action label (one, always the same):** `review_for_refresh`

Before coding the rule, I check the two signals it leans on. Both back the pieces of the rule above:
staleness (`days_since_last_update`) is the signal behind FlyRank's refresh flags, and visibility
(`impressions_90d`) is the signal behind the quick-win logic. Each gets one bucket table, with n,
and one verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.

In [ ]:
# Signal 1 — staleness behind the refresh flags
# Claim: "the longer since a page was updated, the more likely it's currently declining."
tier_order = ["0-30", "31-90", "91-180", "181+"]
sig1 = (
    df.groupby("freshness_tier")["is_declining_label"]
      .agg(decline_rate="mean", n="count")
      .reindex(tier_order)
)
sig1["decline_rate"] = sig1["decline_rate"].round(3)
print(sig1)
print("base rate:", round(df["is_declining_label"].mean(), 3))

                decline_rate      n
freshness_tier                     
0-30                   0.511  20480
31-90                  0.589    175
91-180                 0.611   9171
181+                   0.471    174
base rate: 0.542


**Verdict — MIXED.** Decline rate does climb from `0-30` (0.511) to `91-180` (0.611, n=9,171) —
that part supports the claim, and the bucket is well above the 50-row floor. But `181+` drops back
to 0.471 (n=174). That bucket is thin (barely above the floor) so I won't read much into it alone,
but I also can't call staleness a clean, monotonic signal on this slice. Directionally useful for
"getting stale," not for "the older the better." That's why my rule uses staleness as a gate
(`>= 90 days`), not as the thing it scores by.

In [ ]:
# Signal 2 — visibility/volume behind the quick-win logic
# Claim: "pages with more current search visibility are more likely to be slipping right now."
imp_order = ["low", "moderate", "good", "excellent"]
sig2 = (
    df.groupby("impression_tier")["is_declining_label"]
      .agg(decline_rate="mean", n="count")
      .reindex(imp_order)
)
sig2["decline_rate"] = sig2["decline_rate"].round(3)
print(sig2)
print("base rate:", round(df["is_declining_label"].mean(), 3))

                 decline_rate      n
impression_tier                     
low                     0.454  11248
moderate                0.615  10469
good                    0.586   7205
excellent               0.462   1078
base rate: 0.542


**Verdict — MIXED.** `low` (0.454) and `excellent` (0.462) both sit *below* the base rate; `moderate`
(0.615, n=10,469) and `good` (0.586, n=7,205) sit *above* it — a hump, not a straight line. All four
buckets clear the 50-row floor easily. Reading it plainly: pages with barely any traffic have little
to lose, and the biggest pages are usually stable earners — the mid-size pages are the ones actually
in motion. That's a clearly-explained negative on "more traffic = more risk" as a straight line, and
it's exactly why my rule uses a *band* (`300 <= impressions_90d < 30,000`) instead of a simple
`>= threshold`.

**What this means for the rule:** both signals are real but non-monotonic, so I built the rule to
sit in the band where each signal is actually elevated — not at the extreme end of either one.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# The rule: stale (>= 90 days since update) AND visible (mid-size traffic band).
# No fitted weights, no product flags, no future-window columns -- readable on purpose.
stale = (df["days_since_last_update"] >= 90).astype(int)
visible = ((df["impressions_90d"] >= 300) & (df["impressions_90d"] < 30000)).astype(int)

df["reason_code"] = "stale_visible_decline_risk"
df["action"] = "review_for_refresh"
df["score"] = stale * visible * df["impressions_90d"]

# Rows that don't qualify get score 0 and no action.
df.loc[df["score"] == 0, "action"] = "no_action"
df.loc[df["score"] == 0, "reason_code"] = "not_stale_and_visible"

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in (10, 20, 50):
    p = precision_at_k(df["score"], df["is_declining_label"], k)
    print(f"Precision@{k}: {p:.3f}   (base rate: {df['is_declining_label'].mean():.3f})")

print("\nRows that qualify for review:", (df['score'] > 0).sum(), "of", len(df))

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "score", "reason_code", "action",
            "days_since_last_update", "impressions_90d", "avg_position", "ctr",
            "content_type", "main_intent", "position_tier"]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote work/outputs/baseline_action_score.csv")

Precision@10: 0.500   (base rate: 0.542)
Precision@20: 0.600   (base rate: 0.542)
Precision@50: 0.700   (base rate: 0.542)

Rows that qualify for review: 6745 of 30000
wrote work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Confidence note: do we actually have solid position/CTR data behind this pick, or is it thin?
def confidence_note(row):
    if row["avg_position"] > 0 and pd.notna(row["ctr"]):
        return "high - has position and CTR data"
    return "moderate - missing position or CTR data"

# "What would make it wrong": grounded in the row's own columns, not generic filler.
def what_would_make_it_wrong(row):
    if row["position_tier"] in ("top_3", "page_1"):
        return "wrong if position is still strong and the dip is a short blip, not a real slide"
    if row["content_type"] == "feedly article":
        return "wrong if this is syndicated filler nobody intended to keep evergreen"
    if row["main_intent"] == "transactional":
        return "wrong if conversions/revenue held up even as impressions dipped"
    return "wrong if a refresh is already scheduled or the dip is seasonal, not structural"

top20 = ranked.head(20).copy()
top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(what_would_make_it_wrong, axis=1)
top20["actually_declining"] = top20["content_id"].map(df.set_index("content_id")["is_declining_label"])

review_cols = ["content_id", "action", "reason_code", "confidence_note",
               "what_would_make_it_wrong", "actually_declining"]
print(top20[review_cols].to_string(index=False))

          content_id             action                reason_code                  confidence_note                                                        what_would_make_it_wrong  actually_declining
content_8f39d99dfcfc review_for_refresh stale_visible_decline_risk high - has position and CTR data  wrong if a refresh is already scheduled or the dip is seasonal, not structural                   1
content_b3fccc13da09 review_for_refresh stale_visible_decline_risk high - has position and CTR data  wrong if a refresh is already scheduled or the dip is seasonal, not structural                   0
content_88e1880bd3ab review_for_refresh stale_visible_decline_risk high - has position and CTR data wrong if position is still strong and the dip is a short blip, not a real slide                   0
content_9331e5f7eeab review_for_refresh stale_visible_decline_risk high - has position and CTR data wrong if position is still strong and the dip is a short blip, not a real slide                   0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Weak picks: top-20 rows my rule flagged that were NOT actually declining.
weak = top20[top20["actually_declining"] == 0]
print(f"{len(weak)} of the top 20 were NOT actually declining (flagged but stable/up):\n")
print(weak[["content_id", "reason_code", "what_would_make_it_wrong"]].to_string(index=False))

print()
print("Reading this honestly: the rule is a volume-and-staleness gate, not a decline detector --")
print("it will always catch some pages that are stale-and-visible but simply not slipping right now.")
print(f"Its precision@20 ({precision_at_k(df['score'], df['is_declining_label'], 20):.2f}) beats the")
print(f"base rate ({df['is_declining_label'].mean():.2f}), but it is a directional filter, not a verdict.")

print("\n--- Leakage check ---")
scoring_inputs = ["days_since_last_update", "impressions_90d"]
banned = ["trend_pct", "trend_direction", "is_declining_label"]
print("Columns the score is built from:", scoring_inputs)
print("Confirmed NOT used in scoring:", banned)
print("No product/health-score flags exist in this starter file at all (see data-dictionary.md),")
print("so none could leak in. Both scoring inputs come from the same trailing 90-day window --")
print("no future/last-30-day-vs-prev-30-day comparison window was used as a feature, only as the label.")

8 of the top 20 were NOT actually declining (flagged but stable/up):

          content_id                reason_code                                                        what_would_make_it_wrong
content_b3fccc13da09 stale_visible_decline_risk  wrong if a refresh is already scheduled or the dip is seasonal, not structural
content_88e1880bd3ab stale_visible_decline_risk wrong if position is still strong and the dip is a short blip, not a real slide
content_9331e5f7eeab stale_visible_decline_risk wrong if position is still strong and the dip is a short blip, not a real slide
content_afd71ac0a398 stale_visible_decline_risk  wrong if a refresh is already scheduled or the dip is seasonal, not structural
content_593e2753a70c stale_visible_decline_risk wrong if position is still strong and the dip is a short blip, not a real slide
content_00a44e45d37c stale_visible_decline_risk wrong if position is still strong and the dip is a short blip, not a real slide
content_4b05c87e3f08 stale_visible

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.